<h1>Disclaimer</h1>
This is an example script to control the satellite. It is not meant to be run as is, but to be used as a template.

It does not represent a valid, usable or sane procedure to control a satellite in any way.
It is created using random commands that might include dangerous operations.


In [ ]:
# Global Imports

from api_connect.satio_session import SatIOSession  # Managing the connection to sat:io
from GS1_Group1_mission.gs1_group1 import GS1_Group1  # satellite to be controlled

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

<h2>Connecting to sat:io</h2>
The satio session can be instantiated in multiple ways. One can either set a config file in the format of:

```json
{
  "loglevel": "DEBUG",
  "keycloak": {
    "server_url": "https://localhost:8443",
    "realm_name": "test",
    "client_id": "library",
    "verify_ssl": true
  },
  "credentials": {
    "username": "<username>",
    "password": "<password>"
  },
  "api_url": "https://localhost:8081"
}
```

or use environment variables to set the configuration. The session will look for the following environment variables:
```bash
API_CONNECT_VERIFY_SOCKET_SSL=false
API_CONNECT_API_URL=https://localhost:8081
API_CONNECT_KEYCLOAK_CLIENT_ID=library
API_CONNECT_KEYCLOAK_REALM=<realm_name>
API_CONNECT_KEYCLOAK_URL=https://localhost:8443
API_CONNECT_API_VERIFY_SSL=false
API_CONNECT_USERNAME=<your_username>
API_CONNECT_PASSWORD=<your_password>
API_CONNECT_TIMEOUT=10
```


In [ ]:
from dotenv import load_dotenv
from pathlib import Path


# Make sure the .env file exists and is filled correctly
if not load_dotenv(Path("../.env")):
    raise Exception("No .env file found or empty")

# Create a session with SAT.IO using a settings file
# session = SatIOSession(settings_file=Path('cred.json'))

# Create a session with SAT.IO using environment variables
session = SatIOSession()


<h2>Creating a satellite instance</h2>

In [ ]:
# be aware that the satellite instance is independent from the session and is not to be edited. To manage satellites use the api_connect package. 
gS1_Group1_instance = GS1_Group1()
# Be aware that depending on your access rights for the satellite not all features might be available. 

<h2>Activity management</h2>
This section gives examples gow to work with activities

<h3>Activity creation</h3>
This section shows how to create an activity with and without auto commit

In [14]:
with SatIOSession():  # The session needs to be used as a context for satellite interaction
    # auto commit sends the activity to the backend as soon as it is created
    test_activity = gS1_Group1_instance.create_activity(name="TestActivity", description="This is my test activity")

    act = gS1_Group1_instance.get_activity(test_activity.uuid)
    act.auto_commit = True
    assert act.model_dump() == test_activity.model_dump()

    # without auto commit, you need to call the initial_commit method to send the activity to the backend
    test_activity_2 = gS1_Group1_instance.create_activity(
        name="MyActivity", description="This is my activity", auto_commit=False
    )
    test_activity_2.commit()

    # to cleanup we delete the activities again
    test_activity.delete()
    test_activity_2.delete()


<h3>Activity retrieval</h3>
This section shows how to load an activity from SAT.IO

In [15]:
from pydantic_models.activity import ActivityInfoModel

with SatIOSession():
    test_activity = gS1_Group1_instance.create_activity(name="TestActivity", description="This is my test activity")
    
    # Get the list of activities for the satellite
    activities: list[ActivityInfoModel] = gS1_Group1_instance.get_activity_list()

    # load the activity from the backend
    act = gS1_Group1_instance.get_activity(test_activity.uuid)

    # cleanup
    act.delete()

<h3>Activity modification</h3>
This section shows how to load an activity from SAT.IO and modify it

In [16]:
with SatIOSession():
    test_activity = gS1_Group1_instance.create_activity(name="TestActivity", description="This is my test activity")

    # load the activity from the backend
    act = gS1_Group1_instance.get_activity(test_activity.uuid)
    assert act.model_dump() == test_activity.model_dump()

    # To modify the activity simple change its properties
    act.name = "MyNewActivityName"
    act.description = "This is my new activity description"
    
    # cleanup
    act.delete()

<h2>Command management</h2>

In [22]:
import base64
from pydantic_models.value_field import OctetStringModel, MatrixModel
from datetime import timezone, timedelta
from datetime import datetime


with SatIOSession():
    # Create an activity. A command needs to be associated with an activity
    test_activity = gS1_Group1_instance.create_activity(name="TestActivity", description="This is my test activity")
    
    # Create a command. Be aware that setting a release time will release the command at that time. The release time in this example is therefore set to 10 years in the future.
    command = gS1_Group1_instance.Comp_myComponent.Cmd_MyCompCommand(absolute_release_time=datetime.now(tz=timezone.utc) + timedelta(weeks=52*10))
    
    # By adding a command to an activity, the command is associated with the activity
    test_activity.add_command(command)
    
    # It is possible to release a command ASAP
    command2 = gS1_Group1_instance.Comp_myComponent.Cmd_MyCompCommand()
    test_activity.add_command(command2)
    command2.release()
    
    # A command can also be retrieved back from the activity using its uuid or name
    command3 = gS1_Group1_instance.Comp_myComponent.Cmd_MyCompCommand()
    test_activity.add_command(command3)
    command3 = test_activity.get_command(command_uuid=command3.uuid)
    command3 = test_activity.get_commands_by_name(command_name=command3.name)[0]
    command3 = test_activity.get_command(command_index=2)
    
    # A command with parameter can be created too
    time_value = datetime.fromisoformat("2024-09-18T19:56:00+00:00")
    matrix_value = MatrixModel(rows=4, columns=2, values=[1, 2, 3, 4, 1, 2, 3, 4])
    command4 = myTestSat_instance.Cmd_MyTestCommand2(
        myIntParameter=2,
        myFloatParameter=2.0,
        myStringParameter="test",
        myEnumParameter=1,
        myOctetStringParameter=OctetStringModel(value=base64.b64encode(bytes([0xCA, 0xFE, 0xCA, 0xFE]))),
        myTimeParameter=time_value,
        myMatrixParameter=matrix_value,
    )
    test_activity.add_command(command4)

    # cleanup
    test_activity.delete()

<h3>Command order</h3>

In [ ]:
# To insert a command at a specific position we use the 'position' keyword argument when adding it to the activity
with SatIOSession():
    test_activity =  gS1_Group1_instance.create_activity(name="TestActivity", description="This is my test activity")
    # add a initial command to the activity, the position does not matter yet
    command2 = gS1_Group1_instance.Comp_myComponent.Cmd_MyCompCommand()
    command2.name = "Command2"
    test_activity.add_command(command2)
    
    # add a command to the activity at the beginning, values <= 0 add commands to the beginning
    command0 = gS1_Group1_instance.Comp_myComponent.Cmd_MyCompCommand()
    command0.name = "Command0"
    test_activity.add_command(command0, position=0)
    
    # Now the activity has 2 commands with order 'Command0', 'Command2'
    # We add a command to the activity in between the other commands
    command1 = gS1_Group1_instance.Comp_myComponent.Cmd_MyCompCommand()
    command1.name = "Command1"
    test_activity.add_command(command1, position=1)
    
    # Now the activity has 2 commands with order 'Command0', 'Command1', 'Command2'
    
    # We move command2 to the beginning of the activity
    # Note that we are not adding a command but moving an existing one
    test_activity.add_command(command2, position=0)
    
    # cleanup
    test_activity.delete()
# To re-order commands we add them to a different position to the activity 

<h3>Command callbacks</h3>

In [ ]:
from GS1_Group1_mission.command import SdkCommand
from mmops_tools.models import Command as ToolsCommand

with SatIOSession():
    test_activity =  myTestSat_instance.create_activity(name="TestActivity", description="This is my test activity")
    
    def callback(changed_command: SdkCommand) -> bool:
        """Callback for commands"""
        print(f"Command {changed_command.name} changed: {changed_command.state}")
        if changed_command.state.value >= ToolsCommand.State.ALLOW_RELEASE.value:
            print(f"Command {changed_command.name} allowed for release")
            return True
        return False
    # add a initial command to the activity, the position does not matter yet
    command = myTestSat_instance.Comp_myComponent.Cmd_MyCompCommand(change_callback=callback)
    test_activity.add_command(command)
    
    command.release()
    
    # cleanup
    test_activity.delete()

In [29]:
from mmops_tools.models import Command as ToolsCommand

with SatIOSession():
    telemetry = myTestSat_instance.Var_MyIntVariable.fetch(start_time=datetime.now(tz=timezone.utc) - timedelta(hours=1), end_time=datetime.now(tz=timezone.utc))